# 🧬 Pipeline Completo: Análisis de Expresión Diferencial en Cáncer Uterino

Este cuaderno ejecuta todo el pipeline del proyecto, desde la limpieza de datos hasta la estandarización de los genes diferencialmente expresados (DEGs).

## 📋 Contenido

1. **Requisitos y Configuración**
2. **Análisis DESeq2 (R)**
3. **Procesamiento de Resultados**
4. **Estandarización de IDs con MyGene**
5. **Verificación de Resultados**
6. **Visualización con Diagrama de Venn**

---

## 📦 Requisitos del Sistema

### Librerías de Python

Asegúrate de tener instaladas las siguientes librerías de Python:

```bash
pip install pandas
pip install mygene
pip install openpyxl
```

### Librerías de R

Para ejecutar los análisis DESeq2, necesitas tener instaladas las siguientes librerías de R:

#### Paquetes CRAN:
```r
install.packages(c(
  "tidyverse",
  "here",
  "jsonlite",
  "glue",
  "fs",
  "RColorBrewer",
  "plotly",
  "corrplot",
  "viridis",
  "VennDiagram",
  "ggrepel"
))
```

#### Paquetes Bioconductor:
```r
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c(
  "GEOquery",
  "DESeq2",
  "edgeR",
  "SummarizedExperiment",
  "apeglm",
  "EnhancedVolcano",
  "pheatmap",
  "ComplexHeatmap",
  "clusterProfiler",
  "enrichplot",
  "msigdbr",
  "fgsea",
  "org.Hs.eg.db",
  "AnnotationDbi",
  "biomaRt",
  "GSVA",
  "vsn"
))
```

**Nota:** Los scripts R del proyecto incluyen instalación automática de paquetes faltantes, pero es recomendable instalarlos manualmente primero para evitar tiempos de espera largos.

---

## 1️⃣ Configuración Inicial

Importamos las librerías necesarias y verificamos la estructura del proyecto.

In [ ]:
import os
import subprocess
import pandas as pd
from pathlib import Path
import glob
import sys

# Directorio base del proyecto
BASE_DIR = Path.cwd()
print(f"📁 Directorio base: {BASE_DIR}")

# Verificar directorios principales
dirs_to_check = ['analysis', 'DATA', 'results', 'standard']
for dir_name in dirs_to_check:
    dir_path = BASE_DIR / dir_name
    exists = "✅" if dir_path.exists() else "❌"
    print(f"{exists} {dir_name}/")

### Datasets a Procesar

Definimos los datasets que vamos a analizar:

In [ ]:
# Datasets a procesar
DATASETS = [
    'GSE21656',
    'GSE102787',
    'GSE131565',
    'GSE173201',
    'GSE179661',
    'GSE197561',
    'GSE223827',
    'GSE285498'
]

print("📊 Datasets a procesar:")
for i, dataset in enumerate(DATASETS, 1):
    print(f"  {i}. {dataset}")

---

## 2️⃣ Análisis DESeq2 (R)

Ejecutamos los scripts R de análisis de expresión diferencial para cada dataset.

**Nota**: Este paso puede tardar varios minutos dependiendo del tamaño de los datasets.

In [ ]:
def run_r_pipeline(gse_id):
    """
    Ejecuta el pipeline R de DESeq2 para un dataset específico.
    """
    # Buscar scripts R en el directorio de análisis
    analysis_dir = BASE_DIR / 'analysis' / gse_id
    
    if not analysis_dir.exists():
        print(f"⚠️  No existe directorio de análisis para {gse_id}")
        return False
    
    # Buscar archivos .R en el directorio
    r_scripts = list(analysis_dir.glob('*.R'))
    
    if not r_scripts:
        print(f"⚠️  No se encontraron scripts R para {gse_id}")
        return False
    
    print(f"\n🔬 Ejecutando análisis DESeq2 para {gse_id}...")
    
    for r_script in r_scripts:
        print(f"   📜 Script: {r_script.name}")
        try:
            # Ejecutar script R
            result = subprocess.run(
                ['Rscript', str(r_script)],
                cwd=str(analysis_dir),
                capture_output=True,
                text=True,
                timeout=1800  # 30 minutos timeout
            )
            
            if result.returncode == 0:
                print(f"   ✅ Completado exitosamente")
            else:
                print(f"   ❌ Error en ejecución")
                print(f"   Error: {result.stderr[:200]}")
                
        except subprocess.TimeoutExpired:
            print(f"   ⏱️  Timeout - el script tardó más de 30 minutos")
        except FileNotFoundError:
            print(f"   ❌ Rscript no encontrado. Asegúrate de tener R instalado.")
            return False
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
    
    return True

# Ejecutar análisis R para cada dataset
print("="*60)
print("🧪 FASE 1: ANÁLISIS DESEQ2")
print("="*60)

# Nota: Puedes comentar esta sección si ya tienes los resultados
# for gse_id in DATASETS:
#     run_r_pipeline(gse_id)

print("\n⚠️  NOTA: Los análisis R están comentados por defecto.")
print("   Si necesitas ejecutarlos, descomenta el código anterior.")
print("   Asumiendo que los resultados ya existen en results/...")

---

## 3️⃣ Procesamiento de Resultados

Procesamos los resultados de DESeq2 y los preparamos para la estandarización.

In [ ]:
def run_processing_script(script_name):
    """
    Ejecuta un script de procesamiento Python.
    """
    script_path = BASE_DIR / script_name
    
    if not script_path.exists():
        print(f"⚠️  Script no encontrado: {script_name}")
        return False
    
    print(f"\n🔄 Ejecutando: {script_name}")
    
    try:
        result = subprocess.run(
            ['python', str(script_path)],
            cwd=str(BASE_DIR),
            capture_output=True,
            text=True,
            timeout=1200  # 20 minutos timeout
        )
        
        if result.returncode == 0:
            print(f"✅ Completado exitosamente")
            if result.stdout:
                print(f"Salida:\n{result.stdout[:500]}")
            return True
        else:
            print(f"❌ Error en ejecución")
            if result.stderr:
                print(f"Error: {result.stderr[:500]}")
            return False
            
    except subprocess.TimeoutExpired:
        print(f"⏱️  Timeout - el script tardó más de 20 minutos")
        return False
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return False

print("="*60)
print("🔧 FASE 2: PROCESAMIENTO DE RESULTADOS")
print("="*60)

# Scripts de procesamiento
processing_scripts = [
    'process_gse131565.py',
    'process_gse173201.py',
    'process_gse179661.py',
    'process_gse223827.py',
    'process_gse285498.py'
]

for script in processing_scripts:
    run_processing_script(script)

---

## 4️⃣ Estandarización de IDs con MyGene

Convertimos todos los identificadores de genes a un formato estándar usando la API de MyGene.info.

**Esto incluye:**
- Ensembl ID
- Entrez ID
- Gene Symbol
- log2FoldChange

In [ ]:
print("="*60)
print("🔬 FASE 3: ESTANDARIZACIÓN DE IDS CON MYGENE")
print("="*60)

# Scripts de estandarización
standardization_scripts = [
    'standardize_gse21656_mygene.py',
    'standardize_gse102787_mygene.py',
    'standardize_gse131565_mygene.py',
    'standardize_gse173201_mygene.py',
    'standardize_gse179661_mygene.py',
    'standardize_gse197561_mygene.py',
    'standardize_gse223827_mygene.py',
    'standardize_gse285498_mygene.py'
]

standardization_results = {}

for script in standardization_scripts:
    gse_id = script.replace('standardize_', '').replace('_mygene.py', '').upper()
    success = run_processing_script(script)
    standardization_results[gse_id] = success

# Resumen de resultados
print("\n" + "="*60)
print("📊 RESUMEN DE ESTANDARIZACIÓN")
print("="*60)

for gse_id, success in standardization_results.items():
    status = "✅ Exitoso" if success else "❌ Falló"
    print(f"{gse_id}: {status}")

---

## 5️⃣ Verificación de Resultados

Verificamos que los archivos estandarizados se hayan creado correctamente.

In [ ]:
print("="*60)
print("🔍 FASE 4: VERIFICACIÓN DE RESULTADOS")
print("="*60)

# Buscar todos los archivos _standard.csv
standard_dir = BASE_DIR / 'standard'
standard_files = list(standard_dir.glob('**/*_standard.csv'))

print(f"\n📁 Archivos estandarizados encontrados: {len(standard_files)}\n")

# Crear un DataFrame con información de cada archivo
file_info = []

for file_path in sorted(standard_files):
    try:
        df = pd.read_csv(file_path)
        
        # Extraer información
        gse_id = file_path.parent.name
        file_name = file_path.name
        num_genes = len(df)
        
        # Contar genes up/down regulados
        log2fc_col = None
        for col in df.columns:
            if 'log2' in col.lower() and 'fold' in col.lower():
                log2fc_col = col
                break
        
        if log2fc_col:
            up_regulated = (df[log2fc_col] > 0).sum()
            down_regulated = (df[log2fc_col] < 0).sum()
        else:
            up_regulated = 'N/A'
            down_regulated = 'N/A'
        
        # Verificar columnas requeridas
        required_cols = ['Ensembl_ID', 'Entrez_ID', 'Gene_Symbol']
        has_all_cols = all(col in df.columns for col in required_cols)
        
        file_info.append({
            'Dataset': gse_id,
            'Archivo': file_name,
            'Total_Genes': num_genes,
            'Up_Regulated': up_regulated,
            'Down_Regulated': down_regulated,
            'Columnas_OK': '✅' if has_all_cols else '❌'
        })
        
    except Exception as e:
        print(f"❌ Error leyendo {file_path.name}: {str(e)}")

# Mostrar tabla de resultados
if file_info:
    results_df = pd.DataFrame(file_info)
    print(results_df.to_string(index=False))
    
    # Estadísticas generales
    print("\n" + "="*60)
    print("📈 ESTADÍSTICAS GENERALES")
    print("="*60)
    print(f"Total de archivos procesados: {len(results_df)}")
    print(f"Total de genes únicos (aprox): {results_df['Total_Genes'].sum():,}")
    print(f"Archivos con todas las columnas: {(results_df['Columnas_OK'] == '✅').sum()}")
else:
    print("⚠️  No se encontraron archivos estandarizados.")

### Inspección Detallada de un Archivo

Veamos un ejemplo de archivo estandarizado:

In [ ]:
# Seleccionar el primer archivo para inspección
if standard_files:
    sample_file = standard_files[0]
    print(f"📄 Inspeccionando: {sample_file.name}\n")
    
    df_sample = pd.read_csv(sample_file)
    
    print("Columnas:")
    print(df_sample.columns.tolist())
    
    print("\nPrimeras 10 filas:")
    print(df_sample.head(10))
    
    print("\nInformación del DataFrame:")
    print(df_sample.info())
    
    print("\nEstadísticas descriptivas:")
    print(df_sample.describe())
else:
    print("⚠️  No hay archivos para inspeccionar.")

---

## 6️⃣ Visualización con Diagrama de Venn

Finalmente, podemos usar el diagrama de Venn interactivo para comparar los genes entre datasets.

In [ ]:
print("="*60)
print("📊 DIAGRAMA DE VENN INTERACTIVO")
print("="*60)

venn_html = BASE_DIR / 'venn_diagram_interactive.html'

if venn_html.exists():
    print(f"\n✅ Archivo de Venn encontrado: {venn_html.name}")
    print("\n📋 Instrucciones:")
    print("   1. Abre el archivo 'venn_diagram_interactive.html' en tu navegador")
    print("   2. Carga los archivos *_standard.csv desde el directorio 'standard/'")
    print("   3. Asigna nombres personalizados a cada dataset")
    print("   4. Selecciona el tipo de ID (Ensembl_ID, Entrez_ID, Gene_Symbol)")
    print("   5. Filtra por regulación (Todos, Up, Down)")
    print("   6. Genera el diagrama y explora las intersecciones")
    print("   7. Exporta los resultados a CSV si lo deseas")
    
    # Intentar abrir en el navegador
    try:
        print("\n🌐 Abriendo en el navegador...")
        
        # En Windows, usar el comando 'start' es más confiable
        if sys.platform == 'win32':
            os.startfile(str(venn_html))
        else:
            # Para otros sistemas operativos, usar webbrowser
            import webbrowser
            webbrowser.open(f'file://{venn_html.absolute()}')
        
        print("✅ Navegador abierto exitosamente")
        
    except Exception as e:
        print(f"\n⚠️  No se pudo abrir automáticamente: {str(e)}")
        print(f"   Por favor, abre manualmente: {venn_html}")
else:
    print("\n❌ Archivo de Venn no encontrado.")

---

## 🎯 Resumen Final

Este cuaderno ha ejecutado todo el pipeline del proyecto:

In [ ]:
print("="*60)
print("🎉 PIPELINE COMPLETADO")
print("="*60)

print("\n✅ Pasos ejecutados:")
print("   1. ✓ Configuración inicial")
print("   2. ✓ Análisis DESeq2 (R) - Opcional")
print("   3. ✓ Procesamiento de resultados")
print("   4. ✓ Estandarización de IDs con MyGene")
print("   5. ✓ Verificación de resultados")
print("   6. ✓ Preparación para visualización")

print("\n📁 Archivos generados:")
print(f"   - Archivos estandarizados: {len(standard_files)} archivos en standard/")
print("   - Diagrama de Venn: venn_diagram_interactive.html")

print("\n🔬 Datasets procesados:")
for gse_id in DATASETS:
    status = standardization_results.get(gse_id, False)
    symbol = "✅" if status else "⚠️"
    print(f"   {symbol} {gse_id}")

print("\n📊 Próximos pasos sugeridos:")
print("   1. Revisar los archivos *_standard.csv en el directorio 'standard/'")
print("   2. Usar venn_diagram_interactive.html para comparar datasets")
print("   3. Exportar genes de interés para análisis posteriores")
print("   4. Realizar análisis de enriquecimiento funcional")

print("\n" + "="*60)
print("¡Análisis completado exitosamente! 🎊")
print("="*60)

---

## 📝 Notas Adicionales

### Sobre los Archivos Estandarizados

Los archivos `*_standard.csv` contienen:
- **Ensembl_ID**: Identificador estable de Ensembl (ej: ENSG00000000003)
- **Entrez_ID**: Identificador numérico de NCBI (ej: 3)
- **Gene_Symbol**: Símbolo oficial del gen (ej: A2M)
- **log2FoldChange**: Cambio de expresión en escala log2

### Interpretación del log2FoldChange

- **Positivo**: Gen up-regulado (mayor expresión en condición tratada)
- **Negativo**: Gen down-regulado (menor expresión en condición tratada)
- **Magnitud**:
  - log2FC = 1 → 2x más expresión
  - log2FC = 2 → 4x más expresión
  - log2FC = -1 → 2x menos expresión

### Tiempos de Ejecución

- **Scripts R (DESeq2)**: Timeout de 30 minutos por script
- **Scripts Python (procesamiento y estandarización)**: Timeout de 20 minutos por script

Si algún script excede estos tiempos, considera:
1. Ejecutarlo manualmente fuera del notebook
2. Aumentar el timeout en el código
3. Optimizar el script si es posible

### Solución de Problemas

Si algún paso falla:
1. Verifica que tienes instaladas todas las dependencias (pandas, mygene, openpyxl)
2. Asegúrate de que los archivos de entrada existen en los directorios correctos
3. Revisa los mensajes de error para identificar el problema específico
4. Puedes ejecutar los scripts individualmente para debugging

### Contacto

Para preguntas o problemas, consulta el README.md del proyecto.

---

**Última actualización**: Diciembre 2025